In lecture 13 of MCS 320, we look at expression trees, symbolic and numeric evaluation.

# 1. Expression Trees

We will make a random polynomial with 3 terms, of degree 13, with rational coefficients, in the variables x, y, and z.  To see always the same polynomial, we fix the seed of the random number generator.

In [1]:
P.<x,y,z> = QQ[]
set_random_seed(2022)
p = P.random_element(terms=3, degree=13)
show(p)

2/9*x^5*y^6*z^2 - 1/2*x^3*y^6*z - 35/2*x^5*z^4

To get the operator/operands view to work on ``p``, we must convert to a symbolic expression.  While the shortest way is to cast ``p`` into the Symbolic Ring ``SR``, the alternative is to apply *symbolic evaluation*.  We redeclare x, y, and z as variables and evaluate ``p`` symbolically in those new variables to obtain the expression ``q``.

In [2]:
x, y, z = var('x, y, z')
q = p(x=x, y=y, z=z)
print(q, 'has type', type(q))

2/9*x^5*y^6*z^2 - 1/2*x^3*y^6*z - 35/2*x^5*z^4 has type <class 'sage.symbolic.expression.Expression'>


In the symbolic evaluation ``p(x=x, y=y, z=z)`` at the left of each ``=`` are the symbols used to define ``p``.  At the right of each ``=`` are the newly declared symbols.  Observe that ``q`` is now a symbolic expression.

In [3]:
q.operator()

<function add_vararg at 0x7fdf6f6c6170>

In [4]:
q.operands()

[2/9*x^5*y^6*z^2, -1/2*x^3*y^6*z, -35/2*x^5*z^4]

The operator of ``q`` is an addition which takes as operands a list.

In [5]:
q0 = q.operands()[0]
q0.operator()

<function mul_vararg at 0x7fdf6f6c6200>

In [6]:
q0.operands()

[x^5, y^6, z^2, 2/9]

The first term of ``q`` has the multiplication operator with a list of four operands. 

In [7]:
q00 = q0.operands()[0]
q00.operator()

<built-in function pow>

In [8]:
q00.operands()

[x, 5]

The power of the first operand in the the first term of ``q`` is a binary operator, with two operands.

To draw the expression tree, we use a ``LabelledOrderedTree``.  The labels for the children are the operands in the expression.

On the WSL (Window Subsystem for Linux) running Ubuntu 22.04.2 LTS, the following statement is needed for the `ascii_art` to display properly.

In [9]:
sage.typeset.ascii_art.AsciiArt._terminal_width = lambda x: 80

In [10]:
oplabels = [str(op) for op in q.operands()]
operands = [LabelledOrderedTree([], label=op) for op in oplabels]
exptree = LabelledOrderedTree(operands, label='+')
ascii_art(exptree)

  ______________+________________
 /               /              /            
2/9*x^5*y^6*z^2 -1/2*x^3*y^6*z -35/2*x^5*z^4

We now do the same to the leaves.

In [11]:
oplabels = [op.operands() for op in q.operands()]
oplabels

[[x^5, y^6, z^2, 2/9], [x^3, y^6, z, -1/2], [x^5, z^4, -35/2]]

in the definition of ``oplabels`` we applied the ``operands()`` to every operand of ``q``.

In [12]:
term0leaves = [LabelledOrderedTree([], label=op) for op in oplabels[0]]
term0tree = LabelledOrderedTree(term0leaves, label='*')
ascii_art(term0tree)

  _____*______
 /   /   /   /  
x^5 y^6 z^2 2/9

We do the same operation for the other two terms.

In [13]:
term1leaves = [LabelledOrderedTree([], label=op) for op in oplabels[1]]
term1tree = LabelledOrderedTree(term1leaves, label='*')
ascii_art(term1tree)

  ____*_____
 /   /   / /   
x^3 y^6 z -1/2

In [14]:
term2leaves = [LabelledOrderedTree([], label=op) for op in oplabels[2]]
term2tree = LabelledOrderedTree(term2leaves, label='*')
ascii_art(term2tree)

  ___*____
 /   /   /    
x^5 z^4 -35/2

We still have to replace the powers of the variables by trees.

In [15]:
leafx = LabelledOrderedTree([], label='x')
leaf5 = LabelledOrderedTree([], label='5')
nodex5 = LabelledOrderedTree([leafx, leaf5], label='^')
ascii_art(nodex5)

  ^_
 / /
x 5

In [16]:
leafz = LabelledOrderedTree([], label='z')
leaf4 = LabelledOrderedTree([], label='4')
nodez4 = LabelledOrderedTree([leafz, leaf4], label='^')
ascii_art(nodez4)

  ^_
 / /
z 4

In [17]:
newterm2tree = LabelledOrderedTree([nodex5, nodez4, term2leaves[2]], label='*')
ascii_art(newterm2tree)

    ___*____
   /    /  /    
  ^_   ^_ -35/2
 / /  / /
x 5  z 4 

We can defined the other powers similarly.

In [18]:
leafy = LabelledOrderedTree([], label='y')
leaf2 = LabelledOrderedTree([], label='2')
leaf3 = LabelledOrderedTree([], label='3')
leaf4 = LabelledOrderedTree([], label='4')
leaf6 = LabelledOrderedTree([], label='6')
nodex3 = LabelledOrderedTree([leafx, leaf3], label='^')
ascii_art(nodex3)
nodey6 = LabelledOrderedTree([leafy, leaf6], label='^')
ascii_art(nodey6)
nodez2 = LabelledOrderedTree([leafz, leaf2], label='^')
ascii_art(nodez2)

  ^_
 / /
z 2

In [19]:
ascii_art(term0tree)

  _____*______
 /   /   /   /  
x^5 y^6 z^2 2/9

In [20]:
newterm0tree = LabelledOrderedTree([nodex5, nodey6, nodez2, term0leaves[3]], label='*')
ascii_art(newterm0tree)

    _____*_______
   /    /    /  /  
  ^_   ^_   ^_ 2/9
 / /  / /  / /
x 5  y 6  z 2 

In [21]:
ascii_art(term1tree)

  ____*_____
 /   /   / /   
x^3 y^6 z -1/2

In [22]:
newterm1tree = LabelledOrderedTree([nodex3, nodey6, leafz, term1leaves[3]], label='*')
ascii_art(newterm1tree)

    ____*_____
   /    /  / /   
  ^_   ^_ z -1/2
 / /  / /  
x 3  y 6   

Now we can assemble the entire tree.

In [23]:
nodes = [newterm0tree, newterm1tree, newterm2tree]
exptree = LabelledOrderedTree(nodes, label='+')
ascii_art(exptree)

           _________________+__________________
          /                  /                /        
    _____*_______       ____*_____        ___*____
   /    /    /  /      /    /  / /       /    /  /    
  ^_   ^_   ^_ 2/9    ^_   ^_ z -1/2    ^_   ^_ -35/2
 / /  / /  / /       / /  / /          / /  / /
x 5  y 6  z 2       x 3  y 6          x 5  z 4 

# 2. Fast Callable Objects

We encountered fast callable objects before, with the binary expression trees.  In this section we demonstrate why they are *fast* compared to general expressions.

In [24]:
q

2/9*x^5*y^6*z^2 - 1/2*x^3*y^6*z - 35/2*x^5*z^4

In [25]:
q(x=1.0, y=2.0, z=3.0)

-1385.50000000000

In [26]:
f = fast_callable(q, vars=['x','y','z'])

In [27]:
f(1.0, 2.0, 3.0)

-1385.50000000000

In [28]:
timeit('q(x=1.0, y=2.0, z=3.0)')

625 loops, best of 3: 48.7 μs per loop

In [29]:
timeit('f(1.0, 2.0, 3.0)')

625 loops, best of 3: 5.86 μs per loop

Compare the difference in times.  Even for such a small expression as ``q``, the fast callable is much faster.  Observe however, that fast callables are defined for numerical, not for symbolic evaluation.

In [30]:
f.op_list()

[('load_arg', 0),
 ('ipow', 5),
 ('load_arg', 1),
 ('ipow', 6),
 'mul',
 ('load_arg', 2),
 ('ipow', 2),
 'mul',
 ('load_const', 2/9),
 'mul',
 ('load_arg', 0),
 ('ipow', 3),
 ('load_arg', 1),
 ('ipow', 6),
 'mul',
 ('load_arg', 2),
 'mul',
 ('load_const', -1/2),
 'mul',
 'add',
 ('load_arg', 0),
 ('ipow', 5),
 ('load_arg', 2),
 ('ipow', 4),
 'mul',
 ('load_const', -35/2),
 'mul',
 'add',
 'return']

In [31]:
from sage.ext.fast_callable import ExpressionTreeBuilder
etb = ExpressionTreeBuilder(vars=['x','y','z'])
x = etb.var('x')
y = etb.var('y')
z = etb.var('z')
p(x=x, y=y, z=z)

add(add(add(0, mul(2/9, mul(mul(ipow(v_0, 5), ipow(v_1, 6)), ipow(v_2, 2)))), mul(-1/2, mul(mul(ipow(v_0, 3), ipow(v_1, 6)), ipow(v_2, 1)))), mul(-35/2, mul(ipow(v_0, 5), ipow(v_2, 4))))

Observe the binary structure.